# Module 4 Lab 03 - NumPy Quantitative Validation

This notebook uses NumPy arrays for mathematical operations, statistical calculations, outlier checks, and reconciliation checks.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy"])

from pathlib import Path
import numpy as np
import pandas as pd

SAMPLE_CSV = Path.cwd() / "data" / "m4_raw_financial_transactions_sample.csv"

def load_sample_transactions():
    if SAMPLE_CSV.exists():
        return pd.read_csv(SAMPLE_CSV)
    print("Local CSV not found; using built-in fallback sample rows.")
    return pd.DataFrame([
        {"SourceSystem": "CoreBanking", "TransactionReference": "M4-0001", "TransactionDateText": "2026-06-01", "InstitutionCode": "MCB", "CounterpartyName": "Maseru Commercial Bank", "CurrencyCode": "LSL", "AmountText": "15000.00", "TransactionType": "Deposit", "Channel": "Branch", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Payments", "TransactionReference": "M4-0003", "TransactionDateText": "2026/06/02", "InstitutionCode": "CPS", "CounterpartyName": "Cape Payment Services", "CurrencyCode": "ZAR", "AmountText": "25000", "TransactionType": "Transfer", "Channel": "Clearing", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Treasury", "TransactionReference": "M4-0004", "TransactionDateText": "2026-06-03", "InstitutionCode": "CBL", "CounterpartyName": "Central Bank of Lesotho", "CurrencyCode": "USD", "AmountText": "100000.00", "TransactionType": "FX Settlement", "Channel": "SWIFT", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "CoreBanking", "TransactionReference": "M4-0006", "TransactionDateText": None, "InstitutionCode": "MCB", "CounterpartyName": "Maseru Commercial Bank", "CurrencyCode": "LSL", "AmountText": "9100.00", "TransactionType": "Deposit", "Channel": "Branch", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Treasury", "TransactionReference": "M4-0010", "TransactionDateText": "2026-06-08", "InstitutionCode": "CBL", "CounterpartyName": "Central Bank of Lesotho", "CurrencyCode": "usd", "AmountText": "bad_amount", "TransactionType": "FX Settlement", "Channel": "SWIFT", "LoadBatch": "BATCH-202606"},
    ])

raw_df = load_sample_transactions()
raw_df.insert(0, "RawTransactionID", range(1, len(raw_df) + 1))

In [ ]:
# Minimal cleaning so we can validate numeric arrays.
FX_RATES_TO_LSL = {"LSL": 1.0, "ZAR": 1.0, "USD": 18.25, "EUR": 19.75, "GBP": 23.10}
df = raw_df.copy()
df["CurrencyCode"] = df["CurrencyCode"].astype("string").str.strip().str.upper()
df["Amount"] = pd.to_numeric(df["AmountText"], errors="coerce")
df["TransactionDate"] = pd.to_datetime(df["TransactionDateText"], format="mixed", errors="coerce")

valid_mask = df[["TransactionDate", "InstitutionCode", "CurrencyCode", "Amount"]].notna().all(axis=1)
valid_mask &= df["CurrencyCode"].isin(FX_RATES_TO_LSL)
valid_mask &= df["Amount"] >= 0

clean_df = df.loc[valid_mask].copy()
clean_df["FxRateToLSL"] = clean_df["CurrencyCode"].map(FX_RATES_TO_LSL)
clean_df["AmountLSL"] = (clean_df["Amount"] * clean_df["FxRateToLSL"]).round(2)
display(clean_df[["TransactionReference", "CurrencyCode", "Amount", "AmountLSL"]])

In [ ]:
# NumPy arrays are useful for fast quantitative checks.
amounts = clean_df["Amount"].to_numpy(dtype=float)
amounts_lsl = clean_df["AmountLSL"].to_numpy(dtype=float)

stats = {
    "count": amounts.size,
    "sum_amount": np.round(amounts.sum(), 2),
    "mean_amount": np.round(amounts.mean(), 2),
    "median_amount": np.round(np.median(amounts), 2),
    "std_amount": np.round(amounts.std(ddof=1), 2),
    "min_amount": np.round(amounts.min(), 2),
    "max_amount": np.round(amounts.max(), 2),
    "sum_amount_lsl": np.round(amounts_lsl.sum(), 2),
}

stats

In [ ]:
# Outlier-style validation using z-scores.
# This is not a business rule by itself; it flags rows worth reviewing.
mean_amount = amounts.mean()
std_amount = amounts.std(ddof=1)
z_scores = (amounts - mean_amount) / std_amount

review_df = clean_df[["TransactionReference", "CurrencyCode", "Amount"]].copy()
review_df["ZScore"] = np.round(z_scores, 2)
review_df["ReviewFlag"] = np.where(np.abs(z_scores) >= 2, "Review", "OK")
display(review_df.sort_values("ZScore", ascending=False))

In [ ]:
# Reconciliation checks: detail totals must match summary totals.
summary_df = clean_df.groupby("CurrencyCode", as_index=False).agg(TotalAmount=("Amount", "sum"))

detail_total = np.round(clean_df["Amount"].to_numpy().sum(), 2)
summary_total = np.round(summary_df["TotalAmount"].to_numpy().sum(), 2)

if detail_total != summary_total:
    raise ValueError("Reconciliation failed: detail total does not equal summary total")

print("Reconciliation passed")
print("Detail total:", detail_total)
print("Summary total:", summary_total)